In [1]:
%%bash

nvidia-smi

Thu Oct 19 12:50:11 2023       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 525.125.06   Driver Version: 525.125.06   CUDA Version: 12.0     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  Tesla V100-PCIE...  Off  | 00000000:8B:00.0 Off |                    0 |
| N/A   39C    P0    43W / 250W |  13484MiB / 32768MiB |      0%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
                                                                               
+-------

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig
from pathlib import Path

In [3]:
content_dir = Path('.').resolve()
MODEL_NAME = '/home/maia/git/maia/backend/models/ruGPT-3.5-13B_8bit'
DEFAULT_MESSAGE_TEMPLATE = "<s>{role}\n{content}</s>\n"
DEFAULT_SYSTEM_PROMPT = "Ты — ruGPT-3.5, русскоязычный автоматический ассистент. Ты создаешь запросы в финансовой и банковской областях для язоковых моделей."

In [4]:
class Conversation:
    def __init__(
            self,
            message_template=DEFAULT_MESSAGE_TEMPLATE,
            system_prompt=DEFAULT_SYSTEM_PROMPT,
            start_token_id=2,
            bot_token_id=46787
    ):
        self.message_template = message_template
        self.start_token_id = start_token_id
        self.bot_token_id = bot_token_id
        self.messages = [{
            "role": "system",
            "content": system_prompt
        }]

    def add_user_message(self, message):
        self.messages.append({
            "role": "user",
            "content": message
        })
    
    def add_bot_message(self, message):
        self.messages.append({
            "role": "bot",
            "content": message
        })
    
    def get_prompt(self, tokenizer):
        final_text = ""
        for message in self.messages:
            message_text = self.message_template.format(**message)
            final_text += message_text
        final_text += tokenizer.decode([self.start_token_id, self.bot_token_id])
        return final_text.strip()

In [5]:
def generate(model, tokenizer, prompt, generation_config):
    data = tokenizer(prompt, return_tensors="pt")
    data = {k: v.to(model.device) for k, v in data.items()}
    output_ids = model.generate(
        **data,
        generation_config=generation_config
    )[0]
    output_ids = output_ids[len(data["input_ids"][0]):]
    output = tokenizer.decode(output_ids, skip_special_tokens=True)
    return output.strip()

Если запускать данную модель, то ее инференс на начальном этапе займет около 30 мин, но после, если перезапускать ячейку кода (например, если нужно перезагружать ядро, то инференс займет около 2 мин)

In [6]:
# Load base model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    load_in_8bit=True,
    torch_dtype=torch.float16,
    device_map="auto"
)
model.eval()

Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50272, 5120)
    (wpe): Embedding(2048, 5120)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-39): 40 x GPT2Block(
        (ln_1): LayerNorm((5120,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Linear8bitLt(
            (lora_dropout): ModuleDict(
              (default): Dropout(p=0.1, inplace=False)
            )
            (lora_A): ModuleDict(
              (default): Linear(in_features=5120, out_features=8, bias=False)
            )
            (lora_B): ModuleDict(
              (default): Linear(in_features=8, out_features=15360, bias=False)
            )
            (lora_embedding_A): ParameterDict()
            (lora_embedding_B): ParameterDict()
            (base_layer): Linear8bitLt(in_features=5120, out_features=15360, bias=True)
          )
          (c_proj): Linear8bitLt(in_features=5120, out_features=5120, bias=True)
          (attn_drop

In [13]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)
generation_config = GenerationConfig.from_pretrained(MODEL_NAME)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [26]:
conversation = Conversation()

In [21]:
user_message = 'Сформируй промпт запрос, который по данным клиента сформирует ему лучшее банковское предложение'

In [22]:
conversation.add_user_message(user_message)
prompt = conversation.get_prompt(tokenizer)

In [23]:
print(prompt)

<s>system
Ты — ruGPT-3.5, русскоязычный автоматический ассистент. Ты создаешь запросы в финансовой и банковской областях для язоковых моделей.</s>
<s>user
Сформируй промпт запрос, который по данным клиента сформирует ему лучшее банковское предложение</s>
<s> bot


In [24]:
# Start conversation
output = generate(
    model=model,
    tokenizer=tokenizer,
    prompt=prompt,
    generation_config=generation_config,
)

In [25]:
print("ruGPT-3.5:", output[3:])
print()

ruGPT-3.5: 
Введите данные клиента: ФИО, адрес, телефон, e-mail, номер карты, дата окончания срока действия карты, сумма кредита, процентная ставка, срок кредитования, ежемесячная плата за кредит, количество дней до погашения задолженности, сумма переплаты, комиссия банка, способ оплаты кредита, контактный телефон. 

На основе этих данных я сформирую вам наилучшее банковское предложение. 

Для получения более подробной информации о предложении вы можете обратиться к нам по телефону +7 (495) 777-11-11 или электронной почте info@moneycentral.ru.

